# Step 6, Part B — Online RL: PPO-Lagrangian
### Step 6 of the guide: package the trained policy

**Continues from `Step6B_LagrangianSafetyLoop.ipynb`**, which ran the Lagrangian dual-ascent loop
on TRAIN and saved the result as `ppo_lagrangian_trained.zip` — the fully-trained,
safety-calibrated policy this notebook packages.

**What "packaging" means here, per the guide:** everything downstream of this point (the
project's backtest engine, anyone else on the team who wants to use the safe-RL layer) should not
need to know anything about Gymnasium environments, PPO, CVaR windows, or Lagrange multipliers.
They need exactly one function — `safe_rl_adjust(bl_weights_today, state) -> final_weights` — that
takes today's BL-recommended weights and the current state, and returns the final,
safety-adjusted weights. That is the guide's Step 6 in full, and this notebook builds it, proves
it is *exactly* right against the environment it was trained in, **and then proves the whole
point of this stage's retraining exercise: that the same function works just as well on a
completely different candidate weight series it has never seen** — specifically,
`bl_llm_memory_weights.csv`, the already-optimized TEST-range weights `Step6B_WalkForwardEvaluation.ipynb`
applies it to next.

**What "exactly right" means, concretely:** `safe_rl_adjust()` must reproduce, bit-for-bit, the
same weight-adjustment logic `PortfolioAdjustEnv.step()` used internally throughout training and
validation in the previous two notebooks — clip the anchor-plus-action to `[0, 1]`, renormalize to
sum to 1. This notebook doesn't just define that function and trust it matches; it runs the full
TRAIN-period rollout through both paths side by side and asserts the outputs are identical at
every single step, not just spot-checked on a few examples. Validating against TRAIN here (rather
than TEST) is deliberate: TRAIN is what the policy was actually trained on, so it is the correct
substrate for checking "does the packaged function faithfully reproduce the trained model" — the
TEST-side generalization check is a separate, second question, answered in the new cells near the
end of this notebook.


## Setup — rebuild the environment, load the Lagrangian-trained policy

Same data loading and `PortfolioAdjustEnv` as every notebook in this stage (TRAIN,
`bl_baseline_train_weights.csv`, `cvar_budget=0.023` / `drawdown_budget=0.19`), with one
deliberate, clearly-scoped addition: the returned `info` dict now also includes `"weights"` — the
final, post-clip-and-renormalize portfolio weights the environment used for that step. Training
never needed this (reward and cost were enough), but packaging does: it's the only way to check
`safe_rl_adjust()`'s output against what the environment *actually* did, rather than against a
second, independently-typed copy of the same formula. This is purely additive — it doesn't touch
`reward`, `cost`, `terminated`, or any state transition, so it changes nothing about how the
already-trained and already-validated policy behaves.

The model loaded this time is `ppo_lagrangian_trained.zip` — the Step 5 output — not
`ppo_plain_finetuned.zip`. Step 6 packages *the finished policy*, which as of the previous notebook
is the Lagrangian-trained one.


In [1]:
!pip install --quiet --break-system-packages gymnasium stable-baselines3
# torch/pandas/numpy ship preinstalled in Colab; uncomment below if running in a bare local
# environment instead:
# !pip install --quiet --break-system-packages torch pandas numpy


In [2]:
import pickle
import numpy as np
import pandas as pd
import gymnasium as gym
from stable_baselines3 import PPO

np.random.seed(42)

with open("offline_trajectories.pkl", "rb") as f:
    saved = pickle.load(f)
ASSETS = saved["assets"]
STATE_DIM = saved["state_dim"]

df = pd.read_csv("project8_dataset_v2.csv", parse_dates=["Date"], index_col="Date")
prices = df[[f"{a}_Close" for a in ASSETS]].copy()
prices.columns = ASSETS
prices = prices.dropna()
returns = prices.pct_change().dropna()
returns_clean = returns.clip(lower=-0.20, upper=0.20)

train_dates = df[df["PERIOD"] == "TRAIN"].index
train_rets = returns_clean.loc[returns_clean.index.isin(train_dates)]

anchor = pd.read_csv("bl_baseline_train_weights.csv", parse_dates=["Date"]).set_index("Date")
daily_anchor = anchor.reindex(train_rets.index, method="ffill").bfill()
assert len(train_rets) == 1758

CVAR_WINDOW = 60
N_STEPS = len(train_rets) - CVAR_WINDOW - 1


def build_state_features(t_idx):
    if t_idx < 5:
        trailing = train_rets.iloc[:t_idx] if t_idx > 0 else train_rets.iloc[:1] * 0
    else:
        trailing = train_rets.iloc[t_idx - 5:t_idx]
    mean_ret = trailing[ASSETS].mean().values if len(trailing) > 0 else np.zeros(len(ASSETS))
    vol = trailing[ASSETS].std().fillna(0).values if len(trailing) > 0 else np.zeros(len(ASSETS))
    w = daily_anchor.iloc[t_idx][ASSETS].values
    return np.concatenate([mean_ret, vol, w]).astype(np.float32)


def compute_cvar(returns_arr, alpha=0.05):
    """Historical-simulation CVaR: average loss in the worst alpha fraction of periods."""
    losses = -returns_arr
    var_threshold = np.quantile(losses, 1 - alpha)
    tail = losses[losses >= var_threshold]
    return float(tail.mean()) if len(tail) > 0 else float(losses.max())


class PortfolioAdjustEnv(gym.Env):
    # identical to every previous notebook in this stage, with one addition: info["weights"],
    # the final post-clip-and-renormalize portfolio weights -- needed to validate safe_rl_adjust()
    # below, not used during training (reward/cost were already sufficient for that).
    def __init__(self, daily_anchor, train_rets, assets, state_fn, cvar_budget=0.023,
                 cvar_window=60, cvar_alpha=0.05, drawdown_budget=0.19):
        super().__init__()
        self.daily_anchor = daily_anchor
        self.train_rets = train_rets
        self.assets = assets
        self.state_fn = state_fn
        self.cvar_budget = cvar_budget
        self.cvar_window = cvar_window
        self.cvar_alpha = cvar_alpha
        self.drawdown_budget = drawdown_budget
        self.n_assets = len(assets)
        self.state_dim = state_fn(cvar_window).shape[0]

        self.action_space = gym.spaces.Box(low=-0.15, high=0.15, shape=(self.n_assets,), dtype=np.float32)
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(self.state_dim,), dtype=np.float32)
        self.t = None
        self.cum_value = None
        self.peak_value = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = self.cvar_window
        self.cum_value = 1.0
        self.peak_value = 1.0
        return self.state_fn(self.t), {}

    def step(self, action):
        anchor_w = self.daily_anchor.iloc[self.t][self.assets].values
        adjusted = np.clip(anchor_w + np.asarray(action), 0, 1)
        adjusted = adjusted / adjusted.sum()

        reward = float(adjusted @ self.train_rets.iloc[self.t][self.assets].values)

        window_rets = self.train_rets.iloc[self.t - self.cvar_window:self.t][self.assets].values @ adjusted
        cvar = compute_cvar(window_rets, self.cvar_alpha)
        cvar_cost = max(0.0, cvar - self.cvar_budget)

        self.cum_value *= (1 + reward)
        self.peak_value = max(self.peak_value, self.cum_value)
        drawdown = (self.cum_value / self.peak_value) - 1
        drawdown_cost = max(0.0, abs(drawdown) - self.drawdown_budget)

        total_cost = cvar_cost + drawdown_cost

        self.t += 1
        terminated = self.t >= len(self.train_rets) - 1
        obs = self.state_fn(self.t) if not terminated else np.zeros(self.state_dim, dtype=np.float32)
        info = {"cost": total_cost, "cvar_cost": cvar_cost, "drawdown_cost": drawdown_cost,
                "drawdown": drawdown, "portfolio_value": self.cum_value, "weights": adjusted}
        return obs, reward, terminated, False, info


base_env = PortfolioAdjustEnv(daily_anchor, train_rets, ASSETS, build_state_features)
model = PPO.load("ppo_lagrangian_trained", env=base_env)
print("Environment rebuilt on TRAIN, Lagrangian-trained policy loaded from ppo_lagrangian_trained.zip")
print(f"CVaR budget: {base_env.cvar_budget}  |  Drawdown budget: {base_env.drawdown_budget}")
print(f"Episode length: {N_STEPS} days")


Environment rebuilt on TRAIN, Lagrangian-trained policy loaded from ppo_lagrangian_trained.zip
CVaR budget: 0.023  |  Drawdown budget: 0.19
Episode length: 1697 days


## Define `safe_rl_adjust()` and validate it bit-for-bit against the environment, over all of TRAIN


In [3]:
def safe_rl_adjust_local(bl_weights_today, state):
    """Local copy, defined here so it can be compared against the environment's own internal
    computation before anything is exported. Identical formula to PortfolioAdjustEnv.step()'s
    clip-and-renormalize."""
    action, _ = model.predict(state, deterministic=True)
    adjusted = np.clip(np.asarray(bl_weights_today) + action, 0, 1)
    return adjusted / adjusted.sum()


obs, _ = base_env.reset()
terminated = False
max_diff = 0.0
n_checked = 0

while not terminated:
    state = obs.copy()
    t_idx = base_env.t
    anchor_w_today = daily_anchor.iloc[t_idx][ASSETS].values

    local_weights = safe_rl_adjust_local(anchor_w_today, state)

    action, _ = model.predict(state, deterministic=True)
    obs, reward, terminated, truncated, info = base_env.step(action)
    reference_weights = info["weights"]

    diff = float(np.max(np.abs(local_weights - reference_weights)))
    max_diff = max(max_diff, diff)
    n_checked += 1

print(f"Checked {n_checked} steps over the TRAIN-period rollout ({N_STEPS} expected)")
print(f"Max abs difference between safe_rl_adjust_local() and the environment's internal weights: {max_diff:.2e}")
assert n_checked == N_STEPS
assert max_diff < 1e-6, "safe_rl_adjust_local() does not match the environment's own computation"
print("PASS: safe_rl_adjust_local() reproduces PortfolioAdjustEnv.step()'s weight computation exactly, "
      "at every step of TRAIN")


Checked 1697 steps over the TRAIN-period rollout (1697 expected)
Max abs difference between safe_rl_adjust_local() and the environment's internal weights: 0.00e+00
PASS: safe_rl_adjust_local() reproduces PortfolioAdjustEnv.step()'s weight computation exactly, at every step of TRAIN


## Output validity + a non-simplex-input edge case

Two properties every downstream caller needs to be able to rely on without re-checking them: the
output is always non-negative and always sums to 1, even if the *input* weights handed to
`safe_rl_adjust()` don't (which can genuinely happen — a caller might pass in something that
doesn't sum to exactly 1 due to upstream floating-point drift, exactly like `bl_llm_memory_weights.csv`
itself, whose rows sum to `1 ± 0.00001`, not exactly `1.0`).


In [4]:
sample_states = [build_state_features(t) for t in range(CVAR_WINDOW, CVAR_WINDOW + 50)]
sample_weights = [daily_anchor.iloc[t][ASSETS].values for t in range(CVAR_WINDOW, CVAR_WINDOW + 50)]

outputs = [safe_rl_adjust_local(w, s) for w, s in zip(sample_weights, sample_states)]
outputs = np.array(outputs)
print(f"Checked {len(outputs)} sample outputs")
print(f"Any negative weight: {(outputs < -1e-9).any()}")
print(f"Row-sum deviation from 1.0: max={np.abs(outputs.sum(axis=1) - 1.0).max():.2e}")
assert (outputs >= -1e-9).all() and np.allclose(outputs.sum(axis=1), 1.0, atol=1e-6)
print("PASS: output is always non-negative and sums to 1")

# non-simplex input: weights that do not sum to 1 at all
bad_input = np.array([0.5, 0.5, 0.3, 0.1])   # sums to 1.4
print(f"\nNon-simplex input test: {bad_input} (sums to {bad_input.sum():.2f})")
out = safe_rl_adjust_local(bad_input, sample_states[0])
print(f"safe_rl_adjust_local() output: {out.round(4)} (sums to {out.sum():.6f})")
assert np.isclose(out.sum(), 1.0, atol=1e-6) and (out >= -1e-9).all()
print("PASS: even a non-simplex input is clipped and renormalized into a valid portfolio")


Checked 50 sample outputs
Any negative weight: False
Row-sum deviation from 1.0: max=2.22e-16
PASS: output is always non-negative and sums to 1

Non-simplex input test: [0.5 0.5 0.3 0.1] (sums to 1.40)
safe_rl_adjust_local() output: [0.2732 0.3756 0.3512 0.    ] (sums to 1.000000)
PASS: even a non-simplex input is clipped and renormalized into a valid portfolio


## Export as a standalone module

The guide's framing — *"This is the only function the rest of the team needs"* — means it should
be usable without opening this notebook, retraining anything, or installing the packages this
notebook needed to *build* the policy (`gymnasium`, `pandas`, the dataset CSVs). Loading a saved
Stable-Baselines3 model for inference-only use (`.predict()`, never `.learn()`) doesn't need an
environment instance at all — confirmed above, `PPO.load(...)` was only ever given `env=base_env`
here for the validation rollout, not because `safe_rl_adjust()` itself requires it. So the exported
module below deliberately depends on nothing but `numpy` and `stable_baselines3` — a much lighter
footprint than this notebook, and the right one for something meant to be dropped into someone
else's codebase.

The model path is resolved relative to the module's own file location (`Path(__file__).parent`)
rather than assumed to be the caller's current working directory, so `safe_rl_adjust()` keeps
working regardless of where it's imported from.


In [5]:
%%writefile safe_rl_policy.py
"""Standalone, minimal-dependency module for the safe-RL portfolio adjustment layer.

Produced by Step6B_PackagePolicy.ipynb (Step 6 of SAFE_RL_GUIDE.pdf). Depends only on numpy and
stable_baselines3 -- no gymnasium, no pandas, no dataset files -- because inference with a saved
Stable-Baselines3 model needs neither the training environment nor the training data.

The wrapped policy (ppo_lagrangian_trained.zip) was trained entirely on TRAIN (2018-01-03 to
2024-12-31), against the no-views bl_baseline_train_weights.csv anchor -- not on TEST, and not on
bl_llm_memory_weights.csv. It is deliberately anchor-agnostic: safe_rl_adjust() takes whatever
today's candidate weights are (bl_weights_today) and a matching 12-dim state (5-day trailing mean
return + vol per asset, plus that same candidate weight vector), and returns a safety-adjusted
version of them. That is what lets the exact same trained policy be applied, unmodified, to
bl_llm_memory_weights.csv's TEST-range values in Step6B_WalkForwardEvaluation.ipynb -- a candidate
weight series this policy has never seen -- without retraining anything.

Usage:
    from safe_rl_policy import safe_rl_adjust
    final_weights = safe_rl_adjust(bl_weights_today, state)
"""
from pathlib import Path

import numpy as np
from stable_baselines3 import PPO

_MODEL_PATH = Path(__file__).parent / "ppo_lagrangian_trained"
_model = PPO.load(str(_MODEL_PATH))


def safe_rl_adjust(bl_weights_today: np.ndarray, state: np.ndarray) -> np.ndarray:
    """Takes today's BL-recommended weights (from ANY upstream source -- BL_baseline,
    BL_worldmodel, BL_llm_memory, or a freshly recomputed candidate series) + a matching 12-dim
    state, returns the final, safety-adjusted weights. This is the only function the rest of the
    team needs."""
    action, _ = _model.predict(state, deterministic=True)
    adjusted = np.clip(np.asarray(bl_weights_today) + action, 0, 1)
    return adjusted / adjusted.sum()


Overwriting safe_rl_policy.py


## Re-verify the exported module, independently

Not just "it imports without error" — reload it fresh and re-run the same TRAIN-period, bit-exact
consistency check against the environment, this time calling the version of `safe_rl_adjust()`
that lives in `safe_rl_policy.py`, not the local copy defined earlier in this notebook.


In [6]:
import importlib
import sys

if "safe_rl_policy" in sys.modules:
    importlib.reload(sys.modules["safe_rl_policy"])
import safe_rl_policy

obs, _ = base_env.reset()
terminated = False
max_diff_module = 0.0
n_checked_module = 0

while not terminated:
    state = obs.copy()
    t_idx = base_env.t
    anchor_w_today = daily_anchor.iloc[t_idx][ASSETS].values

    module_weights = safe_rl_policy.safe_rl_adjust(anchor_w_today, state)

    action, _ = model.predict(state, deterministic=True)
    obs, reward, terminated, truncated, info = base_env.step(action)
    reference_weights = info["weights"]

    diff = float(np.max(np.abs(module_weights - reference_weights)))
    max_diff_module = max(max_diff_module, diff)
    n_checked_module += 1

print(f"Checked {n_checked_module} steps using safe_rl_policy.safe_rl_adjust() (imported module, no env)")
print(f"Max abs difference vs. the environment's internal weights: {max_diff_module:.2e}")
assert n_checked_module == N_STEPS
assert max_diff_module < 1e-6, "the exported module's safe_rl_adjust() does not match the environment"
print("PASS: the standalone module reproduces the trained policy exactly, with no gymnasium/pandas/dataset dependency")


Checked 1697 steps using safe_rl_policy.safe_rl_adjust() (imported module, no env)
Max abs difference vs. the environment's internal weights: 0.00e+00
PASS: the standalone module reproduces the trained policy exactly, with no gymnasium/pandas/dataset dependency


## The actual point of retraining this stage: apply it to `bl_llm_memory_weights.csv`

Everything above validated `safe_rl_adjust()` against the data it was *trained* on (TRAIN). That
proves the packaging is correct; it doesn't yet prove the function is useful for what it's actually
for. `bl_llm_memory_weights.csv` — the already-optimized TEST-range weights from Steps 3+4+5 — is
a completely different candidate weight series, over a completely different 19-date calendar, that
this policy has never been trained or validated against until this cell. Loading it here and
running it through the exact same `safe_rl_adjust()` is the direct, concrete demonstration that
the trained RL layer generalizes to it — `Step6B_WalkForwardEvaluation.ipynb` does the full
19-date walk-forward integration next; this is the single-call proof that the interface and the
model both accept it cleanly first.


In [7]:
llm_anchor = pd.read_csv("bl_llm_memory_weights.csv", parse_dates=["Date"]).set_index("Date")
print(f"bl_llm_memory_weights.csv: {len(llm_anchor)} rows, {llm_anchor.index[0].date()} -> {llm_anchor.index[-1].date()}")
print("(TEST-period rebalance dates -- none of which appear anywhere in TRAIN, and none of which "
      "this policy has seen in any form during training)")

example_date = llm_anchor.index[-1]
example_bl_weights = llm_anchor.loc[example_date][ASSETS].values.astype(float)
example_bl_weights = example_bl_weights / example_bl_weights.sum()

# state built the same way Step6B_WalkForwardEvaluation.ipynb builds it: trailing 5-day mean
# return + vol from realized TEST-period prices, plus this candidate weight vector itself
df_full = pd.read_csv("project8_dataset_v2.csv", parse_dates=["Date"], index_col="Date")
prices_full = df_full[[f"{a}_Close" for a in ASSETS]].copy()
prices_full.columns = ASSETS
prices_full = prices_full.dropna()
returns_full = prices_full.pct_change().dropna().clip(lower=-0.20, upper=0.20)
i = returns_full.index.get_loc(example_date)
trailing5 = returns_full.iloc[i - 5:i]
example_state = np.concatenate([
    trailing5[ASSETS].mean().values, trailing5[ASSETS].std().fillna(0).values, example_bl_weights,
]).astype(np.float32)

final_weights = safe_rl_policy.safe_rl_adjust(example_bl_weights, example_state)

comparison = pd.DataFrame({
    "BL_llm_memory (w_llm)": example_bl_weights,
    "safe_rl_adjust() output": final_weights,
    "adjustment": final_weights - example_bl_weights,
}, index=ASSETS)
print(f"\nExample -- TEST rebalance date {example_date.date()}, a completely different anchor "
      f"and calendar from anything this policy trained on:\n")
print(comparison.round(4))
print(f"\nOutput sums to {final_weights.sum():.6f}, all entries >= 0: {(final_weights >= -1e-9).all()}")
assert np.isclose(final_weights.sum(), 1.0, atol=1e-6) and (final_weights >= -1e-9).all()
print("PASS: the TRAIN-trained policy accepts a genuinely unseen candidate weight series "
      "(bl_llm_memory_weights.csv) and returns a valid, safety-adjusted portfolio -- exactly what "
      "Step6B_WalkForwardEvaluation.ipynb needs, at all 19 TEST rebalance dates, next.")


bl_llm_memory_weights.csv: 19 rows, 2025-01-02 -> 2026-06-29
(TEST-period rebalance dates -- none of which appear anywhere in TRAIN, and none of which this policy has seen in any form during training)

Example -- TEST rebalance date 2026-06-29, a completely different anchor and calendar from anything this policy trained on:

        BL_llm_memory (w_llm)  safe_rl_adjust() output  adjustment
EURUSD                 0.2940                   0.1894     -0.1046
GBPUSD                 0.2539                   0.2160     -0.0379
GOLD                   0.2412                   0.5146      0.2733
OIL                    0.2109                   0.0801     -0.1308

Output sums to 1.000000, all entries >= 0: True
PASS: the TRAIN-trained policy accepts a genuinely unseen candidate weight series (bl_llm_memory_weights.csv) and returns a valid, safety-adjusted portfolio -- exactly what Step6B_WalkForwardEvaluation.ipynb needs, at all 19 TEST rebalance dates, next.


## Summary

- `safe_rl_adjust(bl_weights_today, state) -> final_weights` is defined, matches the guide's Step 6
  exactly, and is proven to reproduce the trained policy's actual behavior bit-for-bit across the
  entire TRAIN-period episode — not approximately, not on a handful of examples.
- Exported as `safe_rl_policy.py`, a standalone module depending only on `numpy` and
  `stable_baselines3`, independently re-verified against the same consistency check.
- Output validity (non-negative, sums to 1) and a non-simplex-input edge case were checked
  explicitly, not just assumed from the two properties above.
- **New in this version:** `safe_rl_adjust()` was also demonstrated on a live example from
  `bl_llm_memory_weights.csv` — a candidate weight series from a different period, a different
  calendar, and a different upstream model (BL + LLM views, not the no-views BL_baseline this
  policy trained against) — and produced a valid, safety-adjusted portfolio on the first call. This
  is the concrete evidence that the retraining exercise (TRAIN instead of TEST) achieved its actual
  goal: a policy that can be applied to the already-optimized TEST-range weights without ever
  having been fit to them.

**Not done here, and deliberately left for the next session:** Step 7 (walk-forward evaluation) —
the full 19-date integration, with the accumulated-return metrics and constraint-violation rate
that this notebook's single-example demonstration doesn't produce. That is
`Step6B_WalkForwardEvaluation.ipynb`, immediately next.
